# Object Detection Metrics with Confidence Intervals

## 1. Setup and Installation

First, ensure you have the required packages installed:

In [1]:
%pip install tqdm PyYAML ultralytics

In [2]:
import time
from pathlib import Path
from infer_ci import MetricEvaluator
from ultralytics import YOLO
import shutil

print("✓ Imports successful")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
✓ Imports successful


In [3]:
!pip install infer-ci

## 2. Load Model and Run Predictions

We'll use a pre-trained YOLOv8 model to generate predictions on the COCO128 validation set.

In [5]:
# Load YOLOv8 nano model (smallest and fastest)
print('Loading YOLOv8n model...')
model = YOLO('yolov8n.pt')  # Automatically downloads if not present
print('✓ Model loaded')

model.val(data="coco128.yaml", epochs=1)

dataset_path = Path('coco128')

# # check dataset structur
def validate_yolo_dataset(
    dataset_path: Path,
    split: str = "train2017",
    fix_structure: bool = False,
    verbose: bool = True,
):
    dataset_path = Path(dataset_path)

    issues = {
        "missing_labels": [],
        "missing_images": [],
        "extra_labels": [],
        "extra_images": [],
        "structure_errors": [],
    }

    if verbose:
        print("\n" + "=" * 70)
        print("YOLO DATASET VALIDATION")
        print("=" * 70)
        print(f"Dataset path: {dataset_path}")
        print(f"Split: {split}")

    if not dataset_path.exists():
        issues["structure_errors"].append(f"Dataset directory not found: {dataset_path}")
        return False, issues

    images_root = dataset_path / "images"
    labels_root = dataset_path / "labels"
    images_split = images_root / split
    labels_split = labels_root / split

    # --------------------------------------------------
    # 🔧 FIX: Move files UP from split folder to root
    # --------------------------------------------------
    if fix_structure:
        if images_split.exists():
            images_root.mkdir(exist_ok=True, parents=True)
            moved = 0
            for img in images_split.iterdir():
                if img.is_file():
                    shutil.move(str(img), str(images_root / img.name))
                    moved += 1
            if verbose:
                print(f"🔧 Moved {moved} images from {images_split} → {images_root}")
            if not any(images_split.iterdir()):
                images_split.rmdir()

        if labels_split.exists():
            labels_root.mkdir(exist_ok=True, parents=True)
            moved = 0
            for lbl in labels_split.glob("*.txt"):
                if lbl.is_file():
                    shutil.move(str(lbl), str(labels_root / lbl.name))
                    moved += 1
            if verbose:
                print(f"🔧 Moved {moved} labels from {labels_split} → {labels_root}")
            if not any(labels_split.iterdir()):
                labels_split.rmdir()

    # --------------------------------------------------
    # Validate final structure
    # --------------------------------------------------
    if not images_root.exists():
        issues["structure_errors"].append(f"Images directory not found: {images_root}")
        return False, issues

    if not labels_root.exists():
        issues["structure_errors"].append(f"Labels directory not found: {labels_root}")
        return False, issues

    if verbose:
        print("\n✓ Directory structure:")
        print(f"  Images: {images_root}")
        print(f"  Labels: {labels_root}")

    # --------------------------------------------------
    # Collect files
    # --------------------------------------------------
    image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    image_files = {}

    for ext in image_exts:
        for img in images_root.glob(f"*{ext}"):
            image_files[img.stem] = img
        for img in images_root.glob(f"*{ext.upper()}"):
            image_files[img.stem] = img

    label_files = {lbl.stem: lbl for lbl in labels_root.glob("*.txt")}

    if verbose:
        print("\n📊 Dataset Statistics:")
        print(f"  Total images: {len(image_files)}")
        print(f"  Total labels: {len(label_files)}")

    image_stems = set(image_files)
    label_stems = set(label_files)

    missing_labels = image_stems - label_stems
    missing_images = label_stems - image_stems

    issues["missing_labels"] = sorted(missing_labels)
    issues["missing_images"] = sorted(missing_images)

    matched = len(image_stems & label_stems)
    is_valid = matched > 0 and not issues["structure_errors"]

    if verbose:
        if missing_labels:
            print(f"\n⚠️ Images without labels: {len(missing_labels)}")
            for stem in sorted(list(missing_labels))[:10]:
                print(f"   - {stem}")

        if missing_images:
            print(f"\n⚠️ Labels without images: {len(missing_images)}")
            for stem in sorted(list(missing_images))[:10]:
                print(f"   - {stem}.txt")

    if verbose:
        print(f"\n{'✓' if is_valid else '❌'} Matched pairs: {matched}")

    return is_valid, issues

validate_yolo_dataset(dataset_path, split='train2017', fix_structure=True, verbose=True)

# Run predictions on COCO128 validation images
print('\nRunning predictions on COCO128...')
print('=' * 50)

start = time.time()

results = model.predict(
    source=dataset_path / 'images' / 'train2017',  # Validation images
    imgsz=640,
    conf=0.25,  # Confidence threshold
    verbose=False
)

elapsed = time.time() - start

print(f"✓ Predictions completed in {elapsed:.2f}s")
print(f"  - Predicted on {len(results)} images")
print(f"  - Average: {elapsed/len(results)*1000:.1f}ms per image")

Loading YOLOv8n model...
✓ Model loaded
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1793.3±487.7 MB/s, size: 44.9 KB)
val: Scanning /content/datasets/coco128/labels/train2017.cache... 126 images, 2 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 128/128 44.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 1.9it/s 4.2s
                   all        128        929      0.639      0.536      0.607      0.446
                person         61        254      0.798      0.681      0.766      0.533
               bicycle          3          6      0.514      0.333      0.314      0.261
                   car         12         46      0.813      0.217      0.265      0.162
            motorcycle          4          5      0.687      0.887      0.898      0

## 3. Compute mAP@0.5:0.95 with Confidence Interval

Now we'll compute the mean Average Precision with a 95% confidence interval using bootstrap resampling.

In [7]:
# Initialize the metric evaluator
evaluate = MetricEvaluator()
print("✓ Metric evaluator initialized")

# Compute mAP@0.5:0.95 with confidence interval
print('\nComputing mAP@0.5:0.95 with 95% CI...')
print('=' * 50)

map_value, (lower, upper) = evaluate.evaluate(
    y_true=str(dataset_path),  # Dataset root directory
    y_pred=results,             # prediction results
    task='detection',
    metric='map',           # available: 'map', 'precision', 'recall'
    method='bootstrap_percentile', # default method 'bootstrap_bca'
    n_resamples=100,  # Use 100 for speed (use 1000+ for production)
    plot=True,  # Create histogram plot
    # plot_per_class=True, # Plot per-class distributions default: False
)

print(f"\n" + "=" * 50)
print(f"RESULTS")
print("=" * 50)
print(f"mAP@0.5:0.95: {map_value:.4f}")
print(f"95% CI: [{lower:.4f}, {upper:.4f}]")
print(f"CI width: {upper - lower:.4f}")
print(f"\n✓ Histogram plot saved to results/ directory")

✓ Metric evaluator initialized

Computing mAP@0.5:0.95 with 95% CI...

  Skipped 2 labels without matching images:
     - image 000000000656 not found
     - image 000000000659 not found

  Skipped 2 predictions without matching ground truth labels:
     - label 000000000250.txt file not found
     - label 000000000508.txt file not found



Bootstrap CI: 100%|██████████| 100/100 [00:09<00:00]


Histogram plot saved to: results/mAP@0.5:0.95_bootstrap_percentile_20261021_143911.png

RESULTS
mAP@0.5:0.95: 0.4912
95% CI: [0.4389, 0.5352]
CI width: 0.0963

✓ Histogram plot saved to results/ directory
